# Semantic Vector Search & AI Article Modeling


### Step 1.1: Load Silver Clean Dataset


In [23]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Ensure working directory is project root
while not (Path.cwd() / 'part1_tech_news_lakehouse').is_dir() and Path.cwd().parent != Path.cwd():
    os.chdir('..')

for p in ['.', 'part1_tech_news_lakehouse']:
    if p not in sys.path:
        sys.path.insert(0, p)

silver_csv_path = 'data/processed/tech_news_clean.csv'
df_silver = pd.read_csv(silver_csv_path)
print(f"Loaded {len(df_silver)} clean articles from {os.path.abspath(silver_csv_path)}")
display(df_silver.head(3))


Loaded 750 clean articles from c:\Users\sivak\OneDrive\Pictures\blank_backup\Projectx\tech-news-arr-warehouse\data\processed\tech_news_clean.csv


,article_id,original_index,company_name_clean,has_company_metadata,title,category,category_clean,author,published_date_clean,published_year,...,summary,url,industry,headquarters,founded_year,employee_count,is_public,stock_ticker,company_age,company_size_category
0,ART0661,660,Airbnb,True,Airbnb Achieves Profitability Milestone,Artificial Intelligence,AI_ML,John Smith,18-01-2024,2024,...,Community contribution aims to accelerate inno...,https://technews.example.com/articles/661,Data Analytics,"London, UK",1999.0,19967.0,False,NaN,25.0,Medium
1,ART0118,117,Airbnb,True,Airbnb Reports Record Revenue Growth,AI & ML,AI_ML,Sam Wilson,27-02-2020,2020,...,International expansion strategy focuses on en...,https://technews.example.com/articles/118,Data Analytics,"London, UK",1999.0,19967.0,False,NaN,21.0,Medium
2,ART0725,724,Airbnb,True,Airbnb CEO Discusses Future of AI,AI/ML,AI_ML,Unknown,08-04-2024,2024,...,The company demonstrated significant advances ...,https://technews.example.com/articles/725,Data Analytics,"London, UK",1999.0,19967.0,False,NaN,25.0,Medium


### Step 1.2: Enforce Schema Types & Inspect Records


In [24]:
int_cols = [
    'original_index', 'published_year', 'published_quarter', 'published_month', 
    'word_count', 'revenue_usd_M', 'founded_year', 'employee_count', 'company_age'
]
for c in int_cols:
    if c in df_silver.columns:
        df_silver[c] = pd.to_numeric(df_silver[c], errors='coerce').astype('Int64')

df_silver['is_public'] = df_silver['is_public'].astype('boolean')
df_silver['has_company_metadata'] = df_silver['has_company_metadata'].astype('boolean')

display(df_silver[['article_id', 'title', 'company_name_clean', 'category_clean', 'published_date_clean', 'revenue_usd_M']].head(4))


,article_id,title,company_name_clean,category_clean,published_date_clean,revenue_usd_M
0,ART0661,Airbnb Achieves Profitability Milestone,Airbnb,AI_ML,18-01-2024,7878
1,ART0118,Airbnb Reports Record Revenue Growth,Airbnb,AI_ML,27-02-2020,2600
2,ART0725,Airbnb CEO Discusses Future of AI,Airbnb,AI_ML,08-04-2024,<NA>
3,ART0504,Airbnb Faces Regulatory Scrutiny,Airbnb,AI_ML,27-11-2023,7800


### Step 2.1: Build Text Corpus & Generate 384-Dim TF-IDF Embeddings


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Concatenate title and summary to form semantic article text
texts = (df_silver['title'].fillna('') + ": " + df_silver['summary'].fillna('')).tolist()

vectorizer = TfidfVectorizer(max_features=384, stop_words='english')
embeddings_matrix = vectorizer.fit_transform(texts).toarray()

print(f"Generated embeddings matrix with shape: {embeddings_matrix.shape}")


Generated embeddings matrix with shape: (750, 152)


### Step 2.2: Define Query Vectorizer & Persist Embeddings Matrix


In [26]:
def embed_query(query_text: str) -> np.ndarray:
    """Transforms a raw text query into a 384-dimensional vector embedding."""
    return vectorizer.transform([query_text]).toarray()[0]

emb_dir = 'part1_tech_news_lakehouse/advanced_semantic_search/data/embeddings'
os.makedirs(emb_dir, exist_ok=True)
emb_file = os.path.join(emb_dir, 'article_embeddings.npy')
np.save(emb_file, embeddings_matrix)
print(f"Persisted embeddings matrix to: {os.path.abspath(emb_file)}")


Persisted embeddings matrix to: c:\Users\sivak\OneDrive\Pictures\blank_backup\Projectx\tech-news-arr-warehouse\part1_tech_news_lakehouse\advanced_semantic_search\data\embeddings\article_embeddings.npy


### Step 2.3: Compute Pairwise Similarities & Top-3 Similar Articles


In [27]:
scores_matrix = cosine_similarity(embeddings_matrix)

top_3_list = []
article_ids = df_silver['article_id'].tolist()

for i in range(len(df_silver)):
    scores = pd.Series(scores_matrix[i], index=article_ids)
    scores = scores.drop(index=article_ids[i])
    top_3 = scores.nlargest(3).index.tolist()
    top_3_list.append(", ".join(top_3))

df_silver['top_similar_articles'] = top_3_list

display(df_silver[['article_id', 'title', 'company_name_clean', 'top_similar_articles']].head(5))


,article_id,title,company_name_clean,top_similar_articles
0,ART0661,Airbnb Achieves Profitability Milestone,Airbnb,"ART0450, ART0314, ART0426"
1,ART0118,Airbnb Reports Record Revenue Growth,Airbnb,"ART0262, ART0140, ART0625"
2,ART0725,Airbnb CEO Discusses Future of AI,Airbnb,"ART0642, ART0448, ART0531"
3,ART0504,Airbnb Faces Regulatory Scrutiny,Airbnb,"ART0398, ART0416, ART0196"
4,ART0271,Airbnb Partners with Major Enterprise Client,Airbnb,"ART0431, ART0727, ART0611"


### Step 3.1: Define Vector Semantic Search Function


In [28]:
def embed_query(query_text: str) -> np.ndarray:
    """Transforms a raw text query into a normalized vector embedding."""
    return vectorizer.transform([query_text]).toarray()[0]

def find_similar_articles(query_text, top_k=5):
    """Finds top-k most semantically similar articles using cosine similarity."""
    q_vec = embed_query(query_text)
    scores = cosine_similarity(embeddings_matrix, [q_vec]).flatten()
    results_df = df_silver[['article_id', 'company_name_clean', 'title', 'category_clean']].copy()
    results_df['similarity_score'] = np.round(scores, 4)
    return results_df.nlargest(top_k, 'similarity_score').reset_index(drop=True)


### Step 3.2: Run Vector Semantic Search Example


In [29]:
query = "machine learning infrastructure and gpu acceleration"
print(f"=== Vector Search Results for: '{query}' ===")
display(find_similar_articles(query, top_k=5))


=== Vector Search Results for: 'machine learning infrastructure and gpu acceleration' ===


,article_id,company_name_clean,title,category_clean,similarity_score
0,ART0661,Airbnb,Airbnb Achieves Profitability Milestone,AI_ML,0.0
1,ART0118,Airbnb,Airbnb Reports Record Revenue Growth,AI_ML,0.0
2,ART0725,Airbnb,Airbnb CEO Discusses Future of AI,AI_ML,0.0
3,ART0504,Airbnb,Airbnb Faces Regulatory Scrutiny,AI_ML,0.0
4,ART0271,Airbnb,Airbnb Partners with Major Enterprise Client,AI_ML,0.0


### Step 4.1: Define Hybrid Search (SQL Metadata Filters + Vector Ranking)


In [30]:
def hybrid_search(query_text=None, industry=None, min_arr_M=None, start_year=None, end_year=None, top_k=5):
    """Combines relational attribute filtering with vector semantic similarity ranking."""
    df_filtered = df_silver.copy()
    if industry:
        df_filtered = df_filtered[df_filtered['industry'] == industry]
    if min_arr_M is not None:
        df_filtered = df_filtered[df_filtered['revenue_usd_M'] >= min_arr_M]
    if start_year:
        df_filtered = df_filtered[df_filtered['published_year'] >= start_year]
    if end_year:
        df_filtered = df_filtered[df_filtered['published_year'] <= end_year]

    if df_filtered.empty:
        return pd.DataFrame()

    if query_text:
        q_vec = embed_query(query_text)
        filtered_embeddings = embeddings_matrix[df_filtered.index]
        scores = cosine_similarity(filtered_embeddings, [q_vec]).flatten()
        df_filtered['similarity_score'] = np.round(scores, 4)
        df_filtered = df_filtered.sort_values(by='similarity_score', ascending=False)

    cols = ['article_id', 'company_name_clean', 'industry', 'published_date_clean', 'revenue_usd_M', 'similarity_score', 'title']
    return df_filtered[cols].head(top_k).reset_index(drop=True)


### Step 4.2: Run Hybrid Search Query


In [31]:
display(hybrid_search(
    query_text="generative AI models and foundational architecture", 
    industry="AI/ML", 
    min_arr_M=50, 
    start_year=2022, 
    end_year=2024, 
    top_k=5
))


,article_id,company_name_clean,industry,published_date_clean,revenue_usd_M,similarity_score,title
0,ART0554,NVIDIA,AI/ML,26-02-2022,17500,0.2508,NVIDIA Announces Breakthrough in Large Languag...
1,ART0680,SpaceX,AI/ML,17-02-2024,8925,0.2504,SpaceX Announces Breakthrough in Large Languag...
2,ART0029,NVIDIA,AI/ML,13-09-2023,40500,0.2485,NVIDIA Announces Breakthrough in Large Languag...
3,ART0121,SpaceX,AI/ML,13-06-2023,7600,0.2422,SpaceX Announces Breakthrough in Large Languag...
4,ART0581,SpaceX,AI/ML,16-09-2024,9520,0.2419,SpaceX Announces Breakthrough in Large Languag...


### Step 5.1: Filter High-Value AI Articles (ARR > $50M, 2022-2024)


In [32]:
ai_mask = (
    ((df_silver['category_clean'] == 'AI_ML') | (df_silver['industry'] == 'AI/ML')) &
    (df_silver['published_year'] >= 2022) &
    (df_silver['published_year'] <= 2024) &
    (df_silver['revenue_usd_M'] > 50)
)

df_ai_enriched = df_silver[ai_mask].copy()
df_ai_enriched['company_name'] = df_ai_enriched['company_name_clean']
df_ai_enriched['published_date'] = df_ai_enriched['published_date_clean']
df_ai_enriched['arr_usd'] = (df_ai_enriched['revenue_usd_M'] * 1_000_000).astype('Int64')
df_ai_enriched['category'] = df_ai_enriched['category_clean']
df_ai_enriched['embedding'] = [np.round(embeddings_matrix[i][:16], 4).tolist() for i in df_ai_enriched.index]

print(f"Matched {len(df_ai_enriched)} AI articles meeting the high-value ARR criteria.")


Matched 124 AI articles meeting the high-value ARR criteria.


### Step 5.2: Assemble Enriched Deliverable Schema & Export CSV


In [33]:
ai_cols = [
    'article_id', 'title', 'company_name', 'published_date', 'category', 
    'arr_usd', 'summary', 'url', 'industry', 'founded_year', 
    'headquarters', 'employee_count', 'is_public', 'stock_ticker', 
    'company_age', 'company_size_category', 'top_similar_articles', 'embedding'
]

ai_articles_enriched = df_ai_enriched[ai_cols].reset_index(drop=True)

output_ai_path = 'data/warehouse/ai_articles_enriched.csv'
os.makedirs('data/warehouse', exist_ok=True)
ai_articles_enriched.to_csv(output_ai_path, index=False)
print(f"Successfully exported {os.path.abspath(output_ai_path)} ({len(ai_articles_enriched)} rows x {len(ai_cols)} cols)")
display(ai_articles_enriched.head())


Successfully exported c:\Users\sivak\OneDrive\Pictures\blank_backup\Projectx\tech-news-arr-warehouse\data\warehouse\ai_articles_enriched.csv (124 rows x 18 cols)


,article_id,title,company_name,published_date,category,arr_usd,summary,url,industry,founded_year,headquarters,employee_count,is_public,stock_ticker,company_age,company_size_category,top_similar_articles,embedding
0,ART0661,Airbnb Achieves Profitability Milestone,Airbnb,18-01-2024,AI_ML,7878000000,Community contribution aims to accelerate inno...,https://technews.example.com/articles/661,Data Analytics,1999,"London, UK",19967,False,NaN,25,Medium,"ART0450, ART0314, ART0426","[0.3181, 0.2961, 0.0, 0.0, 0.3181, 0.0, 0.0, 0..."
1,ART0504,Airbnb Faces Regulatory Scrutiny,Airbnb,27-11-2023,AI_ML,7800000000,Platform improvements deliver enhanced perform...,https://technews.example.com/articles/504,Data Analytics,1999,"London, UK",19967,False,NaN,24,Medium,"ART0398, ART0416, ART0196","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,ART0534,Anthropic Achieves Profitability Milestone,Anthropic,28-02-2023,AI_ML,320000000,Investors show strong confidence in the compan...,https://technews.example.com/articles/534,FinTech,2006,"San Francisco, CA",43747,False,NaN,17,Large,"ART0099, ART0572, ART0493","[0.0, 0.3001, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0...."
3,ART0631,Anthropic Acquires Competitor for Undisclosed Sum,Anthropic,09-01-2024,AI_ML,602000000,Major partnership validates technology and ope...,https://technews.example.com/articles/631,FinTech,2006,"San Francisco, CA",43747,False,NaN,18,Large,"ART0047, ART0076, ART0349","[0.0, 0.0, 0.2926, 0.0, 0.0, 0.0, 0.0, 0.0, 0...."
4,ART0249,Anthropic Reports Record Revenue Growth,Anthropic,17-05-2023,AI_ML,380000000,Strategic acquisition strengthens competitive ...,https://technews.example.com/articles/249,FinTech,2006,"San Francisco, CA",43747,False,NaN,17,Large,"ART0177, ART0175, ART0021","[0.0, 0.0, 0.0, 0.3046, 0.0, 0.0, 0.0, 0.0, 0...."


### Step 5.3: Preview Exported AI Enriched Dataset


In [34]:
display(ai_articles_enriched.head(5))


,article_id,title,company_name,published_date,category,arr_usd,summary,url,industry,founded_year,headquarters,employee_count,is_public,stock_ticker,company_age,company_size_category,top_similar_articles,embedding
0,ART0661,Airbnb Achieves Profitability Milestone,Airbnb,18-01-2024,AI_ML,7878000000,Community contribution aims to accelerate inno...,https://technews.example.com/articles/661,Data Analytics,1999,"London, UK",19967,False,NaN,25,Medium,"ART0450, ART0314, ART0426","[0.3181, 0.2961, 0.0, 0.0, 0.3181, 0.0, 0.0, 0..."
1,ART0504,Airbnb Faces Regulatory Scrutiny,Airbnb,27-11-2023,AI_ML,7800000000,Platform improvements deliver enhanced perform...,https://technews.example.com/articles/504,Data Analytics,1999,"London, UK",19967,False,NaN,24,Medium,"ART0398, ART0416, ART0196","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,ART0534,Anthropic Achieves Profitability Milestone,Anthropic,28-02-2023,AI_ML,320000000,Investors show strong confidence in the compan...,https://technews.example.com/articles/534,FinTech,2006,"San Francisco, CA",43747,False,NaN,17,Large,"ART0099, ART0572, ART0493","[0.0, 0.3001, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0...."
3,ART0631,Anthropic Acquires Competitor for Undisclosed Sum,Anthropic,09-01-2024,AI_ML,602000000,Major partnership validates technology and ope...,https://technews.example.com/articles/631,FinTech,2006,"San Francisco, CA",43747,False,NaN,18,Large,"ART0047, ART0076, ART0349","[0.0, 0.0, 0.2926, 0.0, 0.0, 0.0, 0.0, 0.0, 0...."
4,ART0249,Anthropic Reports Record Revenue Growth,Anthropic,17-05-2023,AI_ML,380000000,Strategic acquisition strengthens competitive ...,https://technews.example.com/articles/249,FinTech,2006,"San Francisco, CA",43747,False,NaN,17,Large,"ART0177, ART0175, ART0021","[0.0, 0.0, 0.0, 0.3046, 0.0, 0.0, 0.0, 0.0, 0...."
